# Make Round 2 Parts

Strategy: build genbank parts that can be uploaded into teselagen.

In [35]:
import re
from typing import List, Tuple
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Data import CodonTable

def parse_seq_id(seq_id: str) -> List[Tuple[str, int, str]]:
    """
    Convert 'A12B_C34D' → [('A', 12, 'B'), ('C', 34, 'D')]
    """
    triplets = seq_id.split('_')
    pattern = re.compile(r'^([A-Z\*])(\d+)([A-Z\*])$')
    parsed = []
    for triplet in triplets:
        m = pattern.match(triplet)
        if not m:
            raise ValueError(f"Malformed triplet '{triplet}' in seq_id")
        parsed.append( (m.group(1), int(m.group(2)), m.group(3)) )
    return parsed


def find_orf(plasmid_seq: Seq, aa_seq: str) -> Tuple[int, int, bool]:
    """
    Return (nt_start, frame, is_reverse) where aa_seq begins.
    frame is 0,1,2 relative to the *found* strand.
    """
    aa_seq = aa_seq.upper()
    # forward strand frames
    for frame in range(3):
        aa_trans = plasmid_seq[frame:].translate(to_stop=False)
        idx = aa_trans.find(aa_seq)
        if idx != -1:
            return frame + idx*3, frame, False
    # reverse-complement frames
    rev = plasmid_seq.reverse_complement()
    for frame in range(3):
        aa_trans = rev[frame:].translate(to_stop=False)
        idx = aa_trans.find(aa_seq)
        if idx != -1:
            # convert rev-comp index back to forward coordinates
            nt_start_in_rev = frame + idx*3
            nt_start_fwd = len(plasmid_seq) - (nt_start_in_rev + len(aa_seq)*3)
            return nt_start_fwd, frame, True
    raise ValueError("Protein sequence not found in any reading frame.")


_std_table = CodonTable.unambiguous_dna_by_name['Standard']

def best_replacement_codon(old_codon: str, new_aa: str) -> str:
    """
    Return a codon for `new_aa` that differs least from `old_codon`.
    """
    new_aa = new_aa.upper()
    if new_aa == '*':
        raise ValueError("Cannot introduce a stop codon as a mutation.")
    candidates = [c for c, aa in _std_table.forward_table.items() if aa == new_aa]
    # hamming distance
    best = min(candidates, key=lambda c: sum(b1 != b2 for b1, b2 in zip(c, old_codon)))
    return best.upper()

def split_plasmid_for_mutations(
        gb_path: str,
        protein_aa: str,
        seq_id: str,
        prefix: str ='',
        suffix: str ='',
    ) -> List[SeqRecord]:
    """
    Return a list of SeqRecords (2N+1 parts) that, when concatenated,
    yield the plasmid carrying the specified amino-acid mutations.
    """
    # ---- 0. Preparations ----------------------------------------------------
    record = SeqIO.read(gb_path, 'genbank')
    plasmid = record.seq.upper()
    muts = parse_seq_id(seq_id)
    nt_cds_start, frame, is_rev = find_orf(plasmid, protein_aa)
    
    # Verify old amino acids
    for wt, pos, _ in muts:
        aa_idx = pos - 1  # convert to 0-based
        if protein_aa[aa_idx].upper() != wt.upper():
            raise ValueError(
                f"Wild-type residue at position {pos} is "
                f"'{protein_aa[aa_idx]}', not '{wt}'.")
    
    # ---- 1. Compute nucleotide indices for each mutation -------------------
    nt_positions = []
    seq_len = len(plasmid)
    for _, pos, _ in muts:
        codon_offset = (pos - 1) * 3
        if is_rev:
            # codon on reverse strand
            start_fwd = nt_cds_start + codon_offset  # still forward coords
            start_fwd = seq_len - (start_fwd + 3)    # flip
        else:
            start_fwd = nt_cds_start + codon_offset
        nt_positions.append(start_fwd)
    
    # Sort by ascending nucleotide coordinate
    combined = sorted(zip(nt_positions, muts), key=lambda x: x[0])
    nt_positions, muts_sorted = zip(*combined)
    
    # ---- 2. Build parts -----------------------------------------------------
    parts: List[SeqRecord] = []
    cursor = 0
    for i, (nt_start, (wt_aa, aa_pos, new_aa)) in enumerate(zip(nt_positions, muts_sorted)):
        # a. unchanged stretch before mutation
        if nt_start < cursor:
            raise ValueError("Overlapping mutations?")
        unchanged = plasmid[cursor:nt_start]
        parts.append(SeqRecord(
            unchanged,
            id=f"{prefix}{2*i+1}{suffix}",
            description="",
            annotations={'molecule_type': 'DNA'}
        ))
        
        # b. replacement codon
        old_codon = str(plasmid[nt_start:nt_start+3])
        new_codon = best_replacement_codon(old_codon, new_aa)
        parts.append(SeqRecord(
            Seq(new_codon),
            id=f"{prefix}{2*i+2}{suffix}",
            description=f"{wt_aa}{aa_pos}{new_aa}",
            annotations={'molecule_type': 'DNA'}
        ))
        
        cursor = nt_start + 3
    
    # c. final tail
    parts.append(SeqRecord(
        plasmid[cursor:],
        id=f"{prefix}{2*len(muts)+1}{suffix}",
        description="",
        annotations={'molecule_type': 'DNA'}
    ))
    
    return parts

In [40]:
def sort_seq_ids(seq_ids):
    return sorted(seq_ids, key=lambda x: (int(x[1:-1]), x[0]))


AP_OplR_WT = 'MPTELSTHGWPQPERQVRWAEAISDTYFPLSLEFAAGQPFDGHLQRWSTPSTPLSLSRLRSSQLGYSRSKAHIGQDHEAFYLVTVPHGSEVHFEQDGHQISCSPGGFIVERGDAPYRFHYGTQNDLWVLKLPERALKANLLGHKRYTRHCFDARQGLGRIFVEQLDLCARHFDTSPPAARHLLLEQASATLLLALQQDERVLGSEGSNLSTLHLTRVEQYVQQNLGDPELSPQTIAGACGLSLRYLHKLFAITPYTLNEWVRLQRLEAVHRQLRDPHCHLAIGELAFRWGFADQAQFTRAFRQQYGCTASEVRATRTH'

AP_OplR_seq_ids = sort_seq_ids('''I252Q
I252R
Y255L
F80A
I252A
F287W
Y255T
R148A
Q296H
N139R
W10L
Q63A
Y255M
Y146L
F80S
F80C
I160L
V269C
N139A
A135W
S51D
Q63G
Q123D
N258S'''.split('\n'))

part_list = []
for ii, seq_id in enumerate(AP_OplR_seq_ids):
    parts = split_plasmid_for_mutations(
        'src/notebooks/jacob/data/plasmids/AP_OplR_WT.gb',
        AP_OplR_WT,
         seq_id,
         prefix='AP_OplR_R2_P',
         suffix=f'_{ii+1:02}'
    )
    part_list.extend(parts)
    for part in parts:
        SeqIO.write(part, f'src/notebooks/jacob/data/round2_parts/{part.id}.gb', 'genbank')


/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


# From Teselagen

In [45]:
import requests


# This script is used to post a design to the Teselagen API.
def post_design(session, design):
    """Fetch notebook entry by id."""
    url = f"{BASE_URL}/designs"
    response = session.post(url, json=design)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []


# This function returns a design JSON example that can be used to post a design to the Teselagen API
def get_design_json_example1():

    example = {
        "allowDuplicates": False,
        "designJson": {
            "assembly_method": "golden gate",
            "columns": [
                {
                    "direction": "forward",
                    "icon": "cds",
                    "name": "column 1",
                    "parts": [{"id": "1asod1"}, {"id": "1asod1"}],
                },
                {
                    "direction": "forward",
                    "icon": "cds",
                    "name": "column 2",
                    "parts": [{"id": "1asod2a"}, {"id": "1asod2b"}],
                },
                {
                    "direction": "forward",
                    "icon": "cds",
                    "name": "column 3",
                    "parts": [{"id": "1asod3a"}, {"id": "1asod3b"}],
                },
            ],
            "layout_type": "list",
            "name": "Design2 From Simple Format",
            "sequences": [
                {
                    "name": "simple seq2",
                    "parts": [
                        {
                            "start": 0,
                            "end": 9,
                            "id": "1asod1",
                            "name": "simple part 1",
                            "strand": 1,
                        },
                        {
                            "start": 10,
                            "end": 19,
                            "id": "1asod2a",
                            "name": "simple part 2a",
                            "strand": 1,
                        },
                        {
                            "start": 10,
                            "end": 23,
                            "id": "1asod2b",
                            "name": "simple part 2b",
                            "strand": 1,
                        },
                        {
                            "start": 20,
                            "end": 29,
                            "id": "1asod3a",
                            "name": "simple part 3a",
                            "strand": 1,
                        },
                        {
                            "start": 24,
                            "end": 29,
                            "id": "1asod3b",
                            "name": "simple part 3b",
                            "strand": 1,
                        },
                    ],
                    "sequence": "gatccacacactctttttctactatctact",
                }
            ],
        },
    }
    return example


BASE_URL = "https://jbei.teselagen.com/tg-api"
USERNAME = "jbr@lbl.gov"  # Replace this with your username
PASSWORD = "5dd21974-c17d-4edd-900f-30630d8c3be9"  # Replace this with your OTP password that you get from the app

session: requests.Session = requests.Session()
session.headers.update(
    {"Content-Type": "application/json", "Accept": "application/json"}
)

# Authenticate and get the token
response: requests.Response = session.put(
    url=f"{BASE_URL}/public/auth",
    json={
        "username": USERNAME,
        "password": PASSWORD,
        "expiresIn": "1d",
    },
)
response.raise_for_status()  # Raise an error if a problem is found
session.headers.update(
    {"x-tg-api-token": response.json()["token"]},  # TOKEN
)
session.headers.pop("Content-Type", None)
del response

# get the example design
example1 = get_design_json_example1()

# post the design
post_design(session, example1)


{'id': '53593690-2bef-44f4-9f28-bafcdc611821'}